In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Materials: prediction intervals — independent coding

Four small assignments. Try the learner version first. Reading a reference answer is not independent completion. All checks are illustrative tests, not a guarantee for every input.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/shape_lab').is_dir())
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shape_lab.evidence_v7 import load_snapshot, conformal_rank, conformal_radius, coverage, overlap_report
from shape_lab.display import show_and_save, save_json


## concrete_slump.C1 — `absolute_errors`

Compute each absolute prediction error while preserving target units.

Predict one result by hand before running the check.

<details><summary>Hint</summary>Subtract matching vectors, then take absolute values.</details>

In [3]:
def absolute_errors(y,prediction):
    """Absolute residuals retain the physical units of the target."""
    y=np.asarray(y,float);p=np.asarray(prediction,float)
    if y.ndim!=1 or p.shape!=y.shape or not y.size or not np.isfinite(y).all() or not np.isfinite(p).all():raise ValueError('Matching finite vectors required')
    return np.abs(y-p)


In [4]:
assert np.allclose(absolute_errors([20,30,40],[22,25,43]),[2,5,3])
print('concrete_slump.C1: reference check passed')

concrete_slump.C1: reference check passed


## concrete_slump.C2 — `interval_bounds`

Return lower and upper prediction bounds for a fixed nonnegative radius.

Predict one result by hand before running the check.

<details><summary>Hint</summary>One radius is used on both sides. Do not refit it on these outcomes.</details>

In [5]:
def interval_bounds(prediction,radius):
    """Closed symmetric intervals. Negative lower strength is not silently clipped."""
    p=np.asarray(prediction,float);r=float(radius)
    if p.ndim!=1 or not p.size or not np.isfinite(p).all() or np.isnan(r) or r<0:raise ValueError('Finite predictions and nonnegative radius required')
    return p-r,p+r


In [6]:
lo,hi=interval_bounds([30.],8.)
assert np.allclose(lo,[22.]) and np.allclose(hi,[38.])
print('concrete_slump.C2: reference check passed')

concrete_slump.C2: reference check passed


## concrete_slump.C3 — `empirical_coverage`

Compute the fraction inside closed lower/upper bounds.

Predict one result by hand before running the check.

<details><summary>Hint</summary>Both inequalities must hold; the boundary counts as covered.</details>

In [7]:
def empirical_coverage(y,lower,upper):
    """Fraction of this finite sample within the supplied closed bounds."""
    y=np.asarray(y,float);lo=np.asarray(lower,float);hi=np.asarray(upper,float)
    if y.ndim!=1 or lo.shape!=y.shape or hi.shape!=y.shape or not y.size or not np.isfinite(y).all() or np.isnan(lo).any() or np.isnan(hi).any() or np.any(lo>hi):raise ValueError('Matching ordered interval bounds required')
    return float(np.mean((lo<=y)&(y<=hi)))


In [8]:
assert empirical_coverage([0,1,2],[-1,0,0],[0,1,1]) == 2/3
print('concrete_slump.C3: reference check passed')

concrete_slump.C3: reference check passed


## concrete_slump.C4 — `mean_baseline`

Predict the training mean for n future cases.

Predict one result by hand before running the check.

<details><summary>Hint</summary>The test outcomes are not an argument to this function.</details>

In [9]:
def mean_baseline(training_y,n):
    """A constant prediction fitted only to training outcomes."""
    y=np.asarray(training_y,float)
    if y.ndim!=1 or not y.size or not np.isfinite(y).all() or not isinstance(n,int) or n<1:raise ValueError('Finite training outcomes and positive count required')
    return np.full(n,y.mean())


In [10]:
assert np.allclose(mean_baseline([2.,6.],3),[4,4,4])
print('concrete_slump.C4: reference check passed')

concrete_slump.C4: reference check passed
